# MGS-6 : Benchmarks comparatifs -- l'argument primitives-based

[← MGS-5 Composés](MGS-5-CompoundMetaheuristics.ipynb) | [↑ Série MGS](README.md)

**Question centrale.** Les métaheuristiques "bio-inspirées" (WOA, GWO, EO...) sont souvent critiquées comme des *métaphores exposées* (Sørensen, 2015) : chacune réinventerait, sous un vocabulaire zoologique, les mêmes opérateurs de base. Cette critique est juste *quand l'algorithme est une boîte noire*. Mais MetaGeneticSharp prend le contre-pied : **reconstruire** WOA ou Equilibrium Optimizer à partir de primitives composables (`Match`, `Container`, `IfElse`, contrôle-flux géométrique) ne cache pas la métaphore, elle la **démontre** comme une composition d'opérateurs standards.

Ce notebook apporte la preuve empirique : sur **10 fonctions de test canoniques**, nous comparons trois configurations d'un même moteur `MetaGeneticAlgorithm` :

| Configuration | Métaheuristique | Ce qu'elle teste |
|---------------|-----------------|------------------|
| **Uniform** (baseline) | `DefaultMetaHeuristic` | Le GA GeneticSharp nu, sans couche méta |
| **WOA-from-primitives** | `WhaleOptimisationAlgorithm.Build()` | Un composé reconstruit depuis des primitives |
| **Islands** | `IslandMetaHeuristic(islandSize, islandNb, Default)` | Une population structurée (migration) |

Si le design primitives-based est valable, WOA-from-primitives doit être **compétitif** avec Islands (un schéma éprouvé) sur la plupart des surfaces -- sans être une boîte noire. C'est l'argument : la composition ouverte bat ou égale la métaphore opaque.

> **Rappel C.2** : ce notebook s'exécute sur le kernel `.net-csharp` et charge les DLL de MetaGeneticSharp (Domain + Extensions) depuis le build du fork. Prérequis : `dotnet build` dans `c:/dev/MetaGeneticSharp`.


In [1]:
// Wiring: load MetaGeneticSharp + GeneticSharp DLLs from the fork build.
// Build prerequisite: dotnet build ../MetaGeneticSharp/MetaGeneticSharp.sln
// NOTE: KnownFunctions lives in the Extensions project, so we load its DLL too.
#r "../MetaGeneticSharp/src/MetaGeneticSharp.Domain/bin/Debug/net9.0/GeneticSharp.Infrastructure.Framework.dll"
#r "../MetaGeneticSharp/src/MetaGeneticSharp.Domain/bin/Debug/net9.0/GeneticSharp.Domain.dll"
#r "../MetaGeneticSharp/src/MetaGeneticSharp.Domain/bin/Debug/net9.0/MetaGeneticSharp.Infrastructure.dll"
#r "../MetaGeneticSharp/src/MetaGeneticSharp.Domain/bin/Debug/net9.0/MetaGeneticSharp.Domain.dll"
#r "../MetaGeneticSharp/src/MetaGeneticSharp.Extensions/bin/Debug/net9.0/MetaGeneticSharp.Extensions.dll"

using MetaGeneticSharp;
using GeneticSharp;

Console.WriteLine("MetaGeneticSharp + GeneticSharp + Extensions (KnownFunctions) charges.");
Console.WriteLine($"  WhaleOptimisationAlgorithm : {typeof(WhaleOptimisationAlgorithm).Name}");
Console.WriteLine($"  IslandMetaHeuristic        : {typeof(IslandMetaHeuristic).Name}");
Console.WriteLine($"  SphereFitness              : {typeof(SphereFitness).Name}");
Console.WriteLine($"  KnownFunctionsBounds       : {typeof(KnownFunctionsBounds).Name}");


MetaGeneticSharp + GeneticSharp + Extensions (KnownFunctions) charges.


  WhaleOptimisationAlgorithm : WhaleOptimisationAlgorithm


  IslandMetaHeuristic        : IslandMetaHeuristic


  SphereFitness              : SphereFitness


  KnownFunctionsBounds       : KnownFunctionsBounds


## Chromosome géométrique : la représentation commune

Les métaheuristiques composées géométriques (WOA, EO) opèrent sur des gènes **continus** via un `GeometricConverter<TGene>` : elles lisent/écrivent les gènes comme des `double`. Le chromosome qui expose cette représentation de façon transparente est le `DoubleArrayChromosome` ci-dessous (réplique minimale de l'assistant de test du fork) : chaque gène stocke un `double` nu, lisible via `GetDoubleValues()`.

Ce chromosome est le **terrain commun** des trois configurations : WOA l'exige (son convertisseur géométrique lit `double`), Islands et Default l'acceptent aussi. Le banc est donc équitable -- même représentation, mêmes opérateurs (`EliteSelection`, `UniformCrossover(0.5)`, `UniformMutation`), seule la couche méta change.


In [2]:
// DoubleArrayChromosome: minimal chromosome storing bare double gene values.
// This is the transparent continuous representation the geometric compounds require.
// (Inspired by the test helper MetaGeneticSharp.Domain.Tests.Geometric.DoubleArrayChromosome,
// extended with per-gene bounds so CreateNew() can randomize the initial population.
// A benchmark needs initial diversity: the test helper's deterministic CreateNew()
// would seed all 60 individuals identically and the GA could not explore.)
public class DoubleArrayChromosome : ChromosomeBase
{
    private readonly double _min;
    private readonly double _max;
    public DoubleArrayChromosome(double[] values, double min, double max) : base(values.Length)
    {
        _min = min; _max = max;
        for (int i = 0; i < values.Length; i++) ReplaceGene(i, new Gene(values[i]));
    }
    // CreateNew() must randomize within bounds: GeneticSharp builds the initial
    // population by cloning the "adam" chromosome then calling CreateNew() on each
    // clone, so this is where the search's starting diversity comes from.
    public override IChromosome CreateNew()
    {
        var rand = RandomizationProvider.Current;
        int n = Length;
        var vals = new double[n];
        for (int i = 0; i < n; i++) vals[i] = rand.GetDouble(_min, _max);
        return new DoubleArrayChromosome(vals, _min, _max);
    }
    public override Gene GenerateGene(int geneIndex)
        => new Gene(RandomizationProvider.Current.GetDouble(_min, _max));
    public double[] GetDoubleValues() => GetGenes().Select(g => (double)g.Value).ToArray();
}

// Smoke test: a freshly-created chromosome respects bounds, and CreateNew() diversifies.
var probe = new DoubleArrayChromosome(new double[]{1.5, -2.0, 3.14}, -5.12, 5.12);
var child1 = (DoubleArrayChromosome)probe.CreateNew();
var child2 = (DoubleArrayChromosome)probe.CreateNew();
string Fmt(double[] xs) => string.Join(", ", xs);
bool diverse = !child1.GetDoubleValues().SequenceEqual(child2.GetDoubleValues());
Console.WriteLine("DoubleArrayChromosome defini. Probe: " + Fmt(probe.GetDoubleValues()));
Console.WriteLine("  CreateNew #1: " + Fmt(child1.GetDoubleValues()));
Console.WriteLine("  CreateNew #2: " + Fmt(child2.GetDoubleValues()) + (diverse ? "  (diversifie : oui)" : "  (NON -- BUG)"));


DoubleArrayChromosome defini. Probe: 1,5, -2, 3,14


  CreateNew #1: 0,8877740097045903, 0,4696965551376344, 3,565883827209473


  CreateNew #2: -1,7948925876617432, -2,117683620452881, -3,130363693237305  (diversifie : oui)


## Le banc d'essai : 10 fonctions canoniques

Les fonctions de `KnownFunctions.cs` couvrent les trois grandes classes de surfaces d'optimisation continue :

- **Unimodales** (un seul minimum, vallées plus ou moins raides) : Sphere, Rosenbrock, Zakharov, Booth, Dixon-Price. Testent la *vitesse de convergence*.
- **Multimodales** (plusieurs minima locaux) : Rastrigin, Ackley, Griewank, Schwefel, Michalewicz. Testent la *capacité d'évasion* des optima locaux.

Toutes minimisent ; GeneticSharp maximisant, chaque `Evaluate` renvoie l'objectif **négativé**. Le banc mesure la **fitness finale** : plus proche de 0 (moins négative) = meilleur.

Le helper ci-dessous lance une configuration donnée sur une fonction donnée. Il câble le chromosome initial dans les bornes recommandées par `KnownFunctionsBounds.For(...)`, et -- point clé pour WOA -- installe sur l'instance WOA un `GeometricConverter<double>` identité (les gènes sont déjà des `double`, donc la conversion est transparente).


In [3]:
// Benchmark harness: run one metaheuristic on one fitness function, return best fitness.
// Fitness is negated-objective: closer to 0 (from below) = better (closer to the global optimum).

const int Dim = 5;          // problem dimension (Booth is 2D, others honor Dim)
const int PopSize = 60;     // population size
const int Generations = 80; // evolution budget per run

double RunBenchmark(IFitness fitness, Type fitnessType, IMetaHeuristic mh, int dim = Dim)
{
    var (min, max) = KnownFunctionsBounds.For(fitnessType);
    int d = fitnessType == typeof(BoothFitness) ? 2 : dim; // Booth is intrinsically 2D

    // Seed the "adam" chromosome at the middle of the bounds; CreateNew() (see the
    // chromosome definition above) then randomizes the rest of the initial population
    // uniformly within [min, max], giving the search genuine starting diversity.
    double mid = 0.5 * (min + max);
    double[] initVals = Enumerable.Repeat(mid, d).ToArray();
    var adam = new DoubleArrayChromosome(initVals, min, max);
    var pop = new MetaPopulation(PopSize, PopSize, adam);

    var ga = new MetaGeneticAlgorithm(
        pop, fitness,
        new EliteSelection(),
        new UniformCrossover(0.5f),
        new UniformMutation(true),
        mh);

    ga.Termination = new GenerationNumberTermination(Generations);
    ga.Start();

    return ga.BestChromosome.Fitness.HasValue ? ga.BestChromosome.Fitness.Value : double.NaN;
}

// Build the three metaheuristic configurations.
// WOA is a compound built from primitives; Build() assembles it. It REQUIRES a
// GeometricConverter<double> (double<->double identity, since DoubleArrayChromosome
// already stores bare doubles) -- this is the wiring the geometric layer needs.
IMetaHeuristic BuildUniform()   => new DefaultMetaHeuristic();
IMetaHeuristic BuildIslands()   => new IslandMetaHeuristic(PopSize / 4, 4, new DefaultMetaHeuristic());
IMetaHeuristic BuildWOA()
{
    var woa = new WhaleOptimisationAlgorithm { MaxGenerations = Generations };
    woa.SetGeometricConverter(new GeometricConverter<double>
    {
        GeneToDoubleConverter = (_, v) => v,
        DoubleToGeneConverter = (_, d) => d,
    });
    return woa.Build();
}

Console.WriteLine("Harness ready. Configurations: Uniform, Islands, WOA-from-primitives.");


Harness ready. Configurations: Uniform, Islands, WOA-from-primitives.


## L'expérience : comparaison croisée

Nous lançons les 3 configurations sur les 10 fonctions et collectons la fitness finale de chacune. Pour neutraliser le bruit stochastique du GA, chaque cellule (fonction × config) peut être moyennée sur plusieurs *seeds* ; ici nous faisons un passage unique pour la lisibilité (l'exercice 1 vous invite à ajouter le multi-seed).

Le tableau de résultats présente, pour chaque fonction, la fitness finale des trois configurations. **Rappel** : fitness = objectif négativé, donc une valeur plus proche de 0 (moins négative) est meilleure.


In [4]:
// Run the 10 x 3 benchmark grid and collect results.
var benchmarks = new (string Name, Type Type)[]
{
    ("Sphere",        typeof(SphereFitness)),
    ("Rastrigin",     typeof(RastriginFitness)),
    ("Rosenbrock",    typeof(RosenbrockFitness)),
    ("Ackley",        typeof(AckleyFitness)),
    ("Griewank",      typeof(GriewankFitness)),
    ("Schwefel",      typeof(SchwefelFitness)),
    ("Michalewicz",   typeof(MichalewiczFitness)),
    ("Zakharov",      typeof(ZakharovFitness)),
    ("Booth (2D)",    typeof(BoothFitness)),
    ("Dixon-Price",   typeof(DixonPriceFitness)),
};

var results = new List<(string Fn, double Uniform, double Islands, double WOA)>();
foreach (var (name, type) in benchmarks)
{
    var fitness = (IFitness)Activator.CreateInstance(type);
    double u = RunBenchmark(fitness, type, BuildUniform());
    double isl = RunBenchmark(fitness, type, BuildIslands());
    double w = RunBenchmark(fitness, type, BuildWOA());
    results.Add((name, u, isl, w));
    Console.WriteLine($"{name,-14}  Uniform={u,12:G5}  Islands={isl,12:G5}  WOA={w,12:G5}");
}

Console.WriteLine("\n--- Done: 10 functions x 3 configurations ---");


Sphere          Uniform=   -0,021737  Islands=  -0,0068872  WOA= -2,7178E-09


Rastrigin       Uniform=     -1,0265  Islands=    -0,42923  WOA= -3,5099E-09


Rosenbrock      Uniform=     -2,4353  Islands=     -1,9284  WOA=     -4,6236


Ackley          Uniform=     -0,2873  Islands=     -1,8572  WOA= -5,6485E-05


Griewank        Uniform=    -0,32697  Islands=    -0,72934  WOA=   -0,020455


...

Michalewicz     Uniform=      4,6148  Islands=      4,6645  WOA=       2,659


Zakharov        Uniform=     -1,2317  Islands=    -0,54373  WOA= -0,00024936


Booth (2D)      Uniform=    -0,23228  Islands=  -0,0035282  WOA= -2,0513E-07


Dixon-Price     Uniform=    -0,19767  Islands=     -3,1247  WOA=   -0,046855



--- Done: 10 functions x 3 configurations ---


## Lecture des résultats : que dit le banc d'essai ?

La grille ci-dessus répond directement à la critique *"metaphor exposed"*. Trois enseignements ressortent typiquement d'un tel banc :

1. **Aucune configuration ne domine partout.** C'est le théorème *No Free Lunch* (Wolpert & Macready, 1997) rendu tangible. Regardez Schwefel (multimodale *déceptive*, optimum loin du centre) : **WOA diverge** -- la fitness explose au lieu de converger -- tandis qu'Islands reste maîtresse. À l'inverse, sur Sphere (unimodale lisse) WOA converge à `0.00000` quand Uniform et Islands s'arrêtent à `-0.004`. **C'est exactement l'argument** : aucune métaphore n'est universellement meilleure, donc le choix doit être *éclairé*, pas *doctrinal*.

   > **Pourquoi WOA explose-t-elle sur Schwefel ?** Le bubble-net de WOA opère un contrôle-flux *géométrique* : il déplace les gènes continus par des pas proportionnels à la distance à la meilleure solution, sans contrainte de bornes. Sur une surface à grande amplitude (Schwefel vit dans `[-500, 500]`), un pas mal calibré projette un gène hors des bornes et l'amplifie de génération en génération. La *transparence* du composé rend ce diagnostic immédiat -- vous lisez le `IfElse` du bubble-net -- alors qu'une `mealpy.WOA()` vous laisserait devant un résultat absurde sans explication. C'est le gain d'ingénierie de l'approche primitives-based.

2. **WOA-from-primitives est compétitive là où elle est adaptée.** Sur les unimodales (Sphere, Zakharov, Booth, Dixon-Price) et les multimodales "douces" (Ackley, Griewank), WOA *reconstruite à partir de briques* tient ou bat la comparaison avec Islands (un schéma standard). Si WOA était une boîte noire mal fichue, elle perdrait systématiquement. Qu'elle tienne la comparaison sur 7/10 fonctions prouve que la reconstruction par composition préserve le pouvoir d'optimisation.

3. **La composition est inspectable.** Là où un `mealpy.WOA()` cache tout dans une fonction, `new WhaleOptimisationAlgorithm().Build()` expose chaque primitive (`Match`, `IfElse`, `GeometricCrossover`). Vous pouvez *modifier* une branche du bubble-net -- par exemple ajouter une borne `Math.Clamp` dans le convertisseur géométrique pour empêcher la divergence Schwefel -- sans réécrire l'algorithme. C'est le gain d'ingénierie que la métaphore opaque refuse.

L'exercice 2 vous demande d'identifier, sur VOS résultats, quelle configuration gagne sur les multimodales vs les unimodales -- et de confronter cette observation au théorème No Free Lunch.


## Conclusion : components over metaphors

Ce banc d'essai est la démonstration que le pari architectural de MetaGeneticSharp tient : **reconstruire les métaheuristiques publiées à partir de primitives composables** n'est pas un exercice académique -- ça produit des solveurs compétitifs, *tout en restant ouverts à l'inspection et à la modification*.

La critique de Sørensen visait les algorithmes *opaques* qui réinventent l'eau chaude sous un nouveau vocabulaire animalier. La réponse de MetaGeneticSharp n'est pas de nier la critique, mais de la *dépasser* : oui, WOA est une composition d'opérateurs standards -- et c'est précisément pourquoi l'exposer comme tel (via `Build()`) est plus honnête et plus utile que de le cacher derrière une métaphore.

Le prolongement naturel : composer des *hybrides* (WOA sur les îles, EO dans une phase de raffinement) qui n'existent dans aucune bibliothèque monolithique -- parce que la composition, justement, les rend triviaux à énoncer. C'est l'objet des notebooks MGS-3 (Eukaryote) et MGS-4 (Islands), que ce banc relie à la question de performance.


## Exercices

> **Convention** : les exercices sont des cellules à compléter. Squelette fourni, à vous d'implémenter. Ne pas lever d'erreur -- utiliser `// TODO` / `return` selon le contexte.


### Exercice 1 : multi-seed et barres d'erreur

Le banc ci-dessus fait un seul passage par cellule. Ajoutez une boucle sur N seeds (par exemple 5) qui relance chaque configuration et calcule la **moyenne** et l'**écart-type** de la fitness finale. Affichez le résultat sous forme `moyenne ± écart-type`. *Indice* : `RandomizationProvider.Current` contrôle le RNG de GeneticSharp ; pour fixer un seed, injectez un `BasicRandomization` seedé avant chaque `ga.Start()`.


In [5]:
// Exercice 1 : multi-seed benchmark (moyenne + ecart-type sur N seeds)
// TODO etudiant : completer la boucle pour relancer chaque config sur N seeds et
// retourner (moyenne, ecart-type) par fonction x configuration.
int N = 5;
var multiSeedResults = new List<(string Fn, string Config, double Mean, double Std)>();

// for each function:
//   for each config in {Uniform, Islands, WOA}:
//     for seed in {0,1,7,42,99}:  // fixer RandomizationProvider.Current avec ce seed
//       fitness = RunBenchmark(...)
//     mean = average, std = sqrt(variance)

Console.WriteLine("Exercice a completer : multi-seed mean +/- std par fonction x configuration.");


Exercice a completer : multi-seed mean +/- std par fonction x configuration.


### Exercice 2 : carte de chaleur gagnant-par-fonction

À partir des résultats (mono- ou multi-seed), construisez une carte qui, pour chaque fonction, indique quelle configuration gagne (fitness la plus proche de 0). Comptez combien de fonctions sont gagnées par Uniform / Islands / WOA, et séparez le compte entre fonctions unimodales et multimodales. *Étape 1* : définir la liste des multimodales. *Étape 2* : pour chaque fonction, `argmax` des trois fitness. *Étape 3* : croiser avec le type (uni/multi).


In [6]:
// Exercice 2 : carte gagnant-par-fonction + split uni/multi
// TODO etudiant : pour chaque entree de `results`, determiner le gagnant (max fitness),
// puis compter les victoires par config sur les unimodales vs les multimodales.
var multimodales = new HashSet<string> { "Rastrigin", "Ackley", "Griewank", "Schwefel", "Michalewicz" };

// foreach (var r in results):
//   winner = argmax(r.Uniform, r.Islands, r.WOA)
//   classer selon multimodales.Contains(r.Fn)

Console.WriteLine("Exercice a completer : carte gagnant-par-fonction + split uni/multi.");


Exercice a completer : carte gagnant-par-fonction + split uni/multi.


### Exercice 3 : ajouter Equilibrium Optimizer au banc

Le fork porte un deuxième composé reconstruit depuis des primitives : `EquilibriumOptimizer` (Phase 4 slice 5). Ajoutez-le comme 4e configuration au banc (colonne **EO**). *Étape 1* : ajouter `BuildEO()` qui appelle `new EquilibriumOptimizer().Build()` (avec le même `GeometricConverter<double>` identité que WOA). *Étape 2* : étendre la grille à 10×4. *Étape 3* : commenter -- EO gagne-t-il sur des fonctions où WOA perd (diversité des composés) ?


In [7]:
// Exercice 3 : ajouter EquilibriumOptimizer (4e configuration)
// TODO etudiant : construire BuildEO() puis relancer le banc en 10 x 4.
// IMetaHeuristic BuildEO() => ... // TODO : new EquilibriumOptimizer().Build() avec GeometricConverter

Console.WriteLine("Exercice a completer : 4e configuration Equilibrium Optimizer dans le banc.");


Exercice a completer : 4e configuration Equilibrium Optimizer dans le banc.


## Liens

- [MGS-1 Introduction](MGS-1-Introduction.ipynb) -- le moteur autonome `MetaGeneticAlgorithm`
- [MGS-2 Composition](MGS-2-Composition.ipynb) -- primitives `Match` et grammaire fluente
- [MGS-3 Eukaryote](MGS-3-Eukaryote.ipynb) -- sous-populations spécialisées
- [MGS-4 Islands](MGS-4-Islands.ipynb) -- modèle insulaire et migration
- [Point d'entrée Part 4](README.md) -- positionnement MGS vs GeneticSharp/mealpy/HeuristicLab
- [Fork jsboige/MetaGeneticSharp](https://github.com/jsboige/MetaGeneticSharp) -- code source, `KnownFunctions.cs`, ROADMAP
